# 🔋 EnerGrid AI - Data Exploration Notebook
## Energy Load Forecasting & Anomaly Detection

**Author:** Surya  
**Date:** 2026-01-25  
**Project:** Mini Project MP2911 - B.Tech CSE(AI)  

---

### Objectives:
1. Load and explore the synthetic energy dataset
2. Analyze patterns and trends
3. Identify anomalies
4. Prepare data for modeling

In [ ]:
# Cell 1: Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('✅ Libraries imported successfully!')

In [ ]:
# Cell 2: Load Data
data_path = Path('../data/raw/energy_consumption_2023.csv')
df = pd.read_csv(data_path, parse_dates=['timestamp'])

print('📊 Dataset Overview:')
print('=' * 60)
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'\nDate Range: {df["timestamp"].min()} to {df["timestamp"].max()}')
print(f'\nMissing Values:')
print(df.isnull().sum())
print(f'\nData Types:')
print(df.dtypes)

In [ ]:
# Cell 3: Basic Statistics
print('📈 Basic Statistics:')
print('=' * 60)
print(df.describe())

# Anomaly statistics
print(f'\n🔴 Anomaly Distribution:')
print(f'Total anomalies: {df["is_anomaly"].sum()}')
print(f'Percentage: {df["is_anomaly"].mean()*100:.2f}%')

In [ ]:
# Cell 4: Time Series Visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Time series of load
axes[0, 0].plot(df['timestamp'], df['load_MW'], linewidth=0.5, alpha=0.7, color='blue')
axes[0, 0].set_title('Energy Load Over Time (2023)', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Load (MW)')
axes[0, 0].grid(True, alpha=0.3)

# Highlight anomalies
anomalies = df[df['is_anomaly'] == 1]
axes[0, 0].scatter(anomalies['timestamp'], anomalies['load_MW'], 
                   color='red', s=15, label='Anomalies', zorder=5, alpha=0.6)
axes[0, 0].legend()

# Daily pattern
df['hour'] = df['timestamp'].dt.hour
hourly_avg = df.groupby('hour')['load_MW'].mean()
axes[0, 1].plot(hourly_avg.index, hourly_avg.values, marker='o', linewidth=2)
axes[0, 1].fill_between(hourly_avg.index, hourly_avg.values, alpha=0.3)
axes[0, 1].set_title('Average Daily Load Pattern', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Hour of Day')
axes[0, 1].set_ylabel('Average Load (MW)')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_xticks(range(0, 24, 3))

# Weekly pattern
df['day_name'] = df['timestamp'].dt.day_name()
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekly_avg = df.groupby('day_name')['load_MW'].mean().reindex(weekday_order)
axes[1, 0].bar(weekly_avg.index, weekly_avg.values, color='teal', alpha=0.7)
axes[1, 0].set_title('Average Load by Day of Week', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Day')
axes[1, 0].set_ylabel('Average Load (MW)')
axes[1, 0].tick_params(axis='x', rotation=45)

# Temperature vs Load
axes[1, 1].scatter(df['temperature_C'], df['load_MW'], alpha=0.3, s=10)
axes[1, 1].set_title('Temperature vs Energy Load', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Temperature (°C)')
axes[1, 1].set_ylabel('Load (MW)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/plots/initial_eda.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 5: Anomaly Analysis
# Separate normal and anomaly data
normal_data = df[df['is_anomaly'] == 0]
anomaly_data = df[df['is_anomaly'] == 1]

print('🔍 Anomaly Analysis:')
print('=' * 60)
print(f'Normal records: {len(normal_data):,}')
print(f'Anomaly records: {len(anomaly_data):,}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot comparison
box_data = [normal_data['load_MW'], anomaly_data['load_MW']]
axes[0].boxplot(box_data, labels=['Normal', 'Anomaly'])
axes[0].set_title('Load Distribution: Normal vs Anomaly', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Load (MW)')
axes[0].grid(True, alpha=0.3)

# Hourly distribution of anomalies
anomaly_hour_dist = anomaly_data['hour_of_day'].value_counts().sort_index()
axes[1].bar(anomaly_hour_dist.index, anomaly_hour_dist.values, color='coral')
axes[1].set_title('Anomaly Distribution by Hour', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Number of Anomalies')
axes[1].set_xticks(range(0, 24, 3))
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/plots/anomaly_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 6: Correlation Analysis
# Calculate correlation matrix
corr_matrix = df[['load_MW', 'temperature_C', 'humidity_percent', 
                  'hour_of_day', 'day_of_week', 'month']].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/plots/correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print('📊 Top Correlations with Load:')
print('=' * 60)
load_corr = corr_matrix['load_MW'].sort_values(ascending=False)
for feature, corr_value in load_corr.items():
    if feature != 'load_MW':
        print(f'{feature:20s}: {corr_value:+.3f}')

## 📈 Analysis Complete!

**Key Findings:**
1. Dataset has clear daily patterns (peak in evening)
2. Weekend consumption is lower than weekdays
3. Temperature shows correlation with load
4. Anomalies are evenly distributed (5% of data)

**Next Steps:**
1. Feature engineering
2. Data preprocessing
3. Model building (CNN-LSTM for forecasting, Autoencoder for anomaly detection)